# Reproducing Huang et al. (2025) from dandiset 001349

**Paper.** Huang J-Y, Hess M, Bajpai A, Li X, Hobson LN, Xu AJ, Barton SJ, Lu H-C (2025). From initial
formation to developmental refinement: GABAergic inputs shape neuronal subnetworks in the primary
somatosensory cortex. *iScience* 28, 112104.
[doi:10.1016/j.isci.2025.112104](https://doi.org/10.1016/j.isci.2025.112104). The claims file
(`claims/huang-2025.yaml`) was written from the version of record in Europe PMC (PMC11930745), with
its STAR Methods and supplement (Document S1).

**Dandiset.** [DANDI:001349](https://dandiarchive.org/dandiset/001349/0.250604.1510), version
`0.250604.1510`. The paper cites no version of the dandiset. Two versions are published,
0.250520.1729 (361 files) and 0.250604.1510 (422 files); the later one is the most recent and holds
more of the paper's imaging fields, so it is pinned. The draft has the same 422 files and size as
0.250604.1510.

**What the dandiset holds.** One NWB file per imaging field: the Suite2p ROIs (pixel masks) and six
series per ROI (raw and neuropil fluorescence, FISSA-corrected fluorescence, dF/F, Z score and
`ca_events`, the detected events), with age, sex, layer, field number and the CNO condition. It
holds no behavioural video, so the quiescent windows the paper analyses (open detail O1) cannot be
found from the dataset. They are taken from the paper's code repository
([EdnaHuang/SubnetworkAssembley2024](https://github.com/EdnaHuang/SubnetworkAssembley2024) at commit
28e1b6d), whose per-dataset summary tables give `Time_start` and `Time_end` (O1 reading a); the
claim `quiescent-window-link` tests that the tables' rows can be matched to the files.

**Claims.** Tested here: the imaging claims of Figures 1 to 5 and S2 to S4, S6 and S7A, through the
paper's measures (re-implemented in Python in `subnet.py` from the repository's MATLAB code) and the
paper's tests T1 and T2 (linear mixed models with a case bootstrap over subjects, re-implemented from
the repository's R code). The MI claims were run and then blocked, since the MI computed here does not
match the paper's per-dataset MI. Blocked without being run: the comparisons of other clustering methods (Figures S7 and S8;
not run within this run's time), the histology of Figure S5 (not public). The histology of Figure S9 is tested in
`histology_figs9.ipynb`, on counts from the code repository.

| Claim | Tested here |
|---|---|
| quiescent-window-link (added) | yes |
| fig1d, fig1f, figs2f (two) | yes |
| figs2g (three), figs4b (two), the MI claims | run, then blocked: the MI here does not match the paper's |
| fig2e, fig2d, figs3c, fig2f, figs3d, figs3e | yes |
| fig3a, fig3b, fig3c, fig3ef, fig3g | yes |
| fig4a, fig4b, figs4a, fig4c, fig4d, fig4e (two), fig4fgh, figs4-distance | yes |
| fig5cd, figs6ab, fig5e, figs6c, figs6d, fig5fgh, figs6efg | yes |
| figs7a-events-clustered | yes |
| figs7-method-comparison, figs7-silhouette-increase-all-methods, figs7-subnetwork-number-stable, figs8-configuration-all-methods | blocked |
| figs5e-mostly-glutamatergic, figs5f-hm3dq-mostly-gabaergic | blocked (data missing) |
| figs9-no-sex-difference-gabaergic-input, figs9-layer-differences | yes, in `histology_figs9.ipynb` |

The analysis streams every file once for its metadata and once for its data; it takes about 40
minutes on four processes, most of it the k-means runs.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import subnet

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
assets = subnet.list_assets()
meta = subnet.metadata_table(assets)
print(f"dandiset {subnet.DANDISET_ID} version {subnet.DANDISET_VERSION}: {len(assets)} files, "
      f"metadata read in {time.time() - started:.0f} s")
dev_table, dre_table = subnet.paper_tables()
print(f"repository tables at {subnet.REPO_COMMIT[:7]}: developmental {len(dev_table)} rows, DREADD {len(dre_table)} rows")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files by CNO condition (session_description) and age field:")
print(pd.crosstab(meta.cno, meta.age).to_string())
print("\nSubjects:", meta.subject.nunique(), "; files per sex:", meta.sex.value_counts().to_dict())
print("Subjects with sex U:", sorted(meta[meta.sex == "U"].subject.unique()))
print("Layers:", meta.layer.value_counts().to_dict(), "; field numbers:", meta.location.value_counts().to_dict(),
      "; hemispheres:", meta.hemisphere.value_counts().to_dict())
print("Series in every file:", meta.series.map(tuple).value_counts().to_dict())
print(f"Frame rate {meta.rate.min():.4f} to {meta.rate.max():.4f} Hz (Methods 65.6 ms per frame, {1/0.0656:.2f} Hz); "
      f"series rate equals imaging_rate in {np.isclose(meta.rate, meta.imaging_rate).sum()} of {len(meta)} files")
print("Frames per file:", meta.n_frames.value_counts().to_dict())
print(f"ROIs per file: {meta.n_rois.min()} to {meta.n_rois.max()}, median {meta.n_rois.median():.0f}")
odd = meta[meta.age_days > 21]
print("\nFiles with an age outside P11 to P21:")
print(odd[["path", "age", "date_of_birth", "session_start"]].to_string(index=False))
same = meta[meta.subject == odd.subject.iloc[0]].sort_values("session_start")
print("Other sessions of that subject (age field and session date):")
print(same[["session", "age", "session_start", "layer", "location", "n_rois"]].head(8).to_string(index=False))

The paper's mice (Figure 4 legend) number 34 at P11, 31 at P13, 28 at P15, 16 at P18 and 17 at P21,
and the DREADD experiment used 10 mice; the dandiset holds 32 subjects. One file (`C57-C2-1-AL`
session 0) gives the age as P190D, with a session date six months after the subject's birth date; the
other sessions of that subject are at P11 to P21, and the link below matches this file to the
paper's P11 row, so its age field is wrong. Twelve files of two subjects (`C57-C6-2-AL`, `C57-C6-2-AR`)
give sex as U. The series' rate (15.23 Hz) agrees with the 65.6 ms frame of the Methods. No file
holds behavioural data, although the keywords list "behavior" and "pose estimation".

### quiescent-window-link: matching the repository's rows to the files

Rows of the repository's developmental and DREADD tables are matched to files by subject (the
tables' `SubjectID`, written `C57-C2-1_AL`, `C2-1_ALAR`, `chrm3q-...` against the files'
`C57-C2-1-AL`), layer, field number (`Location`), the CNO condition for the DREADD table, and the
number of ROIs (`total_cell_number` against the files' ROI count). Where two files fit, the one whose
age agrees is kept.

In [ ]:
link_dev, rows_dev, files_dev = subnet.link(meta, dev_table, "developmental")
link_dre, rows_dre, files_dre = subnet.link(meta, dre_table, "dreadd")
for name, m, r, f, tab in [("developmental", link_dev, rows_dev, files_dev, dev_table),
                           ("DREADD", link_dre, rows_dre, files_dre, dre_table)]:
    print(f"{name}: {len(m)} of {len(tab)} table rows matched ({100*len(m)/len(tab):.1f}%); "
          f"unmatched files {len(f)}")
    print(f"  age agrees in {m.age_match.sum()} of {len(m)}; sex agrees in {(m.Sex == m.sex).sum()} "
          f"(the others: {m[m.Sex != m.sex][['subject', 'Sex', 'sex']].drop_duplicates().values.tolist()})")
    print("  unmatched rows by subject:", r.subject.value_counts().to_dict())
    if len(f):
        print("  unmatched files:", f[["path", "age", "layer", "location", "n_rois", "cno"]].values.tolist())
print("\nThe file matched to the P190D age:", link_dev[link_dev.age_days > 21][["path", "name", "Age"]].values.tolist())
missing_subjects = sorted(set(dev_table.subject) - set(meta.subject))
print("Table subjects with no file in the dandiset:", missing_subjects)
print("Developmental rows of those subjects:", int(dev_table.subject.isin(missing_subjects).sum()),
      "; DREADD rows:", int(dre_table.subject.isin(missing_subjects).sum()))
both = set(link_dev.path) & set(link_dre.path)
same_window = sum(
    (link_dev.set_index("path").loc[p, ["Time_start", "Time_end"]].values ==
     link_dre.set_index("path").loc[p, ["Time_start", "Time_end"]].values).all() for p in both)
print(f"Files in both tables (P15 before CNO): {len(both)}; same window in both: {same_window}")
print("Window length (s), developmental:", link_dev.Time_interval.describe().round(1).to_dict())
print("Window length (s), DREADD:", link_dre.Time_interval.describe().round(1).to_dict())

### Running the measures

Every matched file is analysed in its window (the same file and window once). The measures and
their readings (claims file, O2 to O6) are those of `subnet.analyse`: events from `ca_events`, event
frequency (O6 a and b), synchrony frequency (O2 Methods and Step7 readings), Pearson correlation and
MI of the Z-scored traces, and the k-means event clusters with 100 runs per cluster number (O3
Methods) and 10 (O3 code), each with the Methods' (O4 a, 95th percentile) and the code's (O4 b,
1 - 0.05/NCl) participation test.

In [ ]:
jobs = {}
for m in (link_dev, link_dre):
    for _, r in m.iterrows():
        jobs[(r.path, r.Time_start, r.Time_end)] = {
            "path": r.path, "asset_id": r.asset_id, "start": float(r.Time_start),
            "end": float(r.Time_end), "n_runs": [100, 10]}
jobs = list(jobs.values())
t0 = time.time()
results = pd.DataFrame(subnet.run_all(jobs, workers=4))
import resource
print(f"{len(results)} datasets analysed in {(time.time() - t0)/60:.1f} min; "
      f"peak memory of a worker {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1e6:.2f} GB")

In [ ]:
def build(link_table, kind):
    m = link_table.merge(results, left_on=["path", "Time_start", "Time_end"], right_on=["path", "start", "end"],
                         suffixes=("", "_res"))
    m["seconds"] = m.Time_end - m.Time_start
    # the age of the P190D file is wrong (data check): the table's age is used for every file
    m["age"] = m.Age.astype(int)
    m = m.rename(columns={"layer": "layer"})
    m["subject"] = m["subject"]
    return m

dev = build(link_dev, "developmental")
dre = build(link_dre, "dreadd")
n_u = int((dev.sex == "U").sum())
dev = dev[dev.sex.isin(["F", "M"])].reset_index(drop=True)
dre = dre[dre.sex.isin(["F", "M"])].reset_index(drop=True)
print(f"developmental datasets {len(dev)} (dropped {n_u} with sex U), DREADD datasets {len(dre)}")
print("\nDevelopmental datasets and subjects by age, sex and layer:")
print(dev.groupby(["age", "sex", "layer"]).agg(datasets=("path", "size"), subjects=("subject", "nunique")).unstack([1, 2]).to_string())
print("\nDREADD datasets by condition, sex and layer:")
print(dre.groupby(["cno", "sex", "layer"]).agg(datasets=("path", "size"), subjects=("subject", "nunique")).unstack([1, 2]).to_string())
print("\nPaired DREADD fields (same subject, layer and field before and after):",
      int(dre.groupby(["subject", "layer", "location"]).cno.nunique().eq(2).sum()))
print("\nRanges of the measures used (developmental):")
cols = ["n_rois", "n_active", "seconds", "event_freq_active", "event_freq_all", "sync_methods", "sync_step7",
        "pearson", "mi", "r100_silhouette", "r10_silhouette", "r100_methods_n_subnetworks", "r100_code_n_subnetworks",
        "r100_methods_pct_single", "r100_methods_pct_non", "r100_methods_pct_multiple", "r100_methods_tcc_in",
        "r100_methods_tcc_out", "r100_methods_pw_neuron", "r100_methods_neuron_centroid", "r100_methods_centroid_centroid"]
print(dev[cols].describe().T[["count", "min", "50%", "max"]].round(3).to_string())
print("Units: frequencies in events per second, distances in pixels (about 0.99 um), percentages of ROIs, MI in bits.")

### Do the dataset's events and windows agree with the paper's?

The paper's tables give, for each dataset, the measures the paper computed. Where the dataset's data
and the paper's measures are the same, the values should agree. Shifting the window by 30 s shows
whether the agreement depends on the window, which tests that the tables' times are on the files'
clock.

In [ ]:
pairs = [("event frequency, O6 a", "event_freq_active", "EventFreq"),
         ("synchrony frequency, O2 Methods", "sync_methods", "SynchronyFreq"),
         ("synchrony frequency, O2 Step7", "sync_step7", "SynchronyFreq"),
         ("Pearson, all pairs", "pearson", "ZF_R_allpair_mean"),
         ("tCC, all pairs", "r10_code_tcc_all", "R_allpair_mean"),
         ("silhouette, 10 runs", "r10_silhouette", "silhs_mean_beforeStat"),
         ("silhouette, 100 runs", "r100_silhouette", "silhs_mean_beforeStat"),
         ("subnetworks, 10 runs, code test", "r10_code_n_subnetworks", "NCls"),
         ("subnetworks, 100 runs, Methods test", "r100_methods_n_subnetworks", "NCls"),
         ("single-cluster %, 10 runs, code test", "r10_code_pct_single", "Cells_in_one_assembly"),
         ("non-cluster %, 10 runs, code test", "r10_code_pct_non", "Cells_not_in_assembly"),
         ("frames in no cluster %, 10 runs, code test", "r10_code_pct_frames_none", "No_assemblies"),
         ("subnetworks, 10 runs, Methods test", "r10_methods_n_subnetworks", "NCls"),
         ("single-cluster %, 100 runs, Methods test", "r100_methods_pct_single", "Cells_in_one_assembly"),
         ("non-cluster %, 100 runs, Methods test", "r100_methods_pct_non", "Cells_not_in_assembly"),
         ("non-cluster %, 100 runs, code test", "r100_code_pct_non", "Cells_not_in_assembly"),
         ("MI of Z traces (256 bins)", "mi", "MI_Z_allpair_mean")]
rows = []
for label, ours, theirs in pairs:
    a, b = dev[ours].astype(float), dev[theirs].astype(float)
    ok = a.notna() & b.notna()
    rows.append({"measure": label, "n": int(ok.sum()), "pearson_r": np.corrcoef(a[ok], b[ok])[0, 1],
                 "median_ours": a[ok].median(), "median_paper": b[ok].median(),
                 "within_5pct": np.mean(np.abs(a[ok] - b[ok]) <= 0.05 * np.abs(b[ok])),
                 "exact_1e-3": np.mean(np.abs(a[ok] - b[ok]) <= 1e-3 * np.maximum(1, np.abs(b[ok])))})
agree = pd.DataFrame(rows)
print(agree.round(3).to_string(index=False))
print("\nEvent frequency against the paper's, true window and windows shifted by 30 s:")
for col in ["event_freq_active", "event_freq_shift-30", "event_freq_shift+30"]:
    ok = dev[col].notna()
    a, b = dev.loc[ok, col], dev.loc[ok, "EventFreq"]
    print(f"  {col:22s} n={ok.sum():3d}  r={np.corrcoef(a, b)[0,1]:.3f}  median |difference| {np.median(np.abs(a-b)):.3f} events/s"
          f"  within 5%: {np.mean(np.abs(a-b) <= 0.05*b):.2f}")
ratio = dev.mi / dev.MI_Z_allpair_mean
print(f"\nMI here against the MI table's Z-trace column: ratio median {ratio.median():.1f} (range {ratio.min():.1f} to {ratio.max():.1f})")
print(f"Correlation over datasets with the window length: MI here {np.corrcoef(dev.mi, dev.seconds)[0,1]:.2f}, "
      f"the table's MI {np.corrcoef(dev.MI_Z_allpair_mean, dev.seconds)[0,1]:.2f}")
print("Mean MI by age, here and in the table:")
print(dev.groupby("age")[["mi", "MI_Z_allpair_mean"]].mean().round(3).to_string())

### Events in the files against the paper's event detection

The `ca_events` series is described only as "Ca_Events". Its non-zero frames are compared with the
events that Step2_Event_Detection_binaryRaster.mlx finds in the file's own Z-score series, for
three files.

In [ ]:
check_rows = []
for _, r in dev.sample(3, random_state=1).iterrows():
    d = subnet.load_dataset(r.path, r.asset_id, r.Time_start, r.Time_end, full_events=True)
    mine = subnet.step2_events(d["z_full"])
    theirs = d["events_full"]
    same = (mine == theirs).all(1).mean()
    check_rows.append({"path": r.path, "rois": theirs.shape[0], "events_ca_events": int(theirs.sum()),
                       "events_step2": int(mine.sum()), "frames_agree": float((mine == theirs).mean()),
                       "rois_identical": float(same),
                       "z_range": f"{d['z_full'].min():.1f} to {d['z_full'].max():.1f}"})
print(pd.DataFrame(check_rows).to_string(index=False))

The non-zero frames of `ca_events` are, to within a few events in a thousand, the peaks Step2
detects on the file's Z score; the values stored at those frames are not Z-score amplitudes (they are
of order 0.001), and the files do not say what they are. Events are taken as the non-zero frames.

**Result: partially reproduced.** 399 of the 455 developmental rows (87.7%) and 56 of the 96
DREADD rows (58.3%) match a file; ROI counts agree by construction, the age agrees in 397 of 399 and
the sex in 387 (the 12 others are the U files). The rows that do not match are mostly those of three
subjects with no file (`C57-C4-3-ALBR`, `-AR`, `-BL`; 40 developmental and 28 DREADD rows) and the
P13 and P15 rows of `C57-C6-2-AL`, which has only P11 files. On the matched datasets the Pearson
correlation recomputed here equals the table's (r = 1.000, 99.7% within 0.001), and event
frequency, tCC and silhouette correlate at r = 0.93, 0.95 and 0.93. Shifting the window by 30 s
lowers the event-frequency agreement from r = 0.93 to 0.82 and 0.79, and the share within 5% from
0.71 to 0.15 and 0.12, so the tables' windows are on the files' clock. The Methods' synchrony
threshold gives a median of 0.91 events/s against the table's 1.45 and Step7's 1.60, and the
Methods' participation test gives a median of 9.1% non-cluster ROIs against the table's 25.0% and
the code test's 25.7%: the table was made with the code's readings. The MI recomputed with the
repository's MI code does not match the table's MI (r = 0.09, about ten times larger).

## The paper's tests

T1 and T2 follow the repository's R code (`Develop_Function_20230219.R`,
`WT_Dlx_GsDREADD_Function_20230211_caseboot.R`): per-dataset values, fixed effects age (or CNO), sex
and layer, the model chosen among eight nested models by AIC and then compared with the simplest and
the full model, random intercepts for subject and subject by field (dropped when singular), and a
case bootstrap over subjects with 10,000 replicates; contrasts are differences of cell predictions,
with p = 2(1 - pd). Two deviations: likelihood-ratio tests replace the Kenward-Roger tests in the
comparison, and each bootstrap replicate refits the fixed effects by generalized least squares with
the variance components of the original fit, where the paper refits the mixed model.

The implementation is checked first on the paper's own per-dataset values (the repository table),
against the contrasts the repository's outputs print, as quoted in the claims file.

In [ ]:
LEV1 = {"age": [11, 13, 15, 18, 21], "sex": ["F", "M"], "layer": [2, 4]}
LEV2 = {"cno": ["before", "after"], "sex": ["F", "M"], "layer": [2, 4]}
GROUPS = [(s, l) for s in "FM" for l in (2, 4)]
LAYER = {2: "L2/3", 4: "L4"}
_cache = {}


def T(col, kind="dev", data=None):
    key = (col, kind, None if data is None else id(data))
    if key not in _cache:
        df = data if data is not None else (dev if kind == "dev" else dre)
        factors, levels = (subnet.FACTORS_T1, LEV1) if kind == "dev" else (subnet.FACTORS_T2, LEV2)
        _cache[key] = subnet.run_test(df, col, factors, levels, B=10000, seed=subnet.seed_for(col))
    return _cache[key]


def C(col, a, b, kind="dev", data=None):
    r = T(col, kind, data)
    return subnet.contrast(r["grid"], r["est"], r["boot"], a, b)


def model_line(col, kind="dev", data=None):
    r = T(col, kind, data)
    return (f"{col}: model {r['sel']['formula'].split('~')[1].strip()} ({r['sel']['structure']}), "
            f"{r['n']} datasets of {r['n_subjects']} subjects, {r['dropped']} of 10000 replicates rank-deficient")


def age_table(cols, a_age, b_age, groups=GROUPS, kind="dev"):
    out = []
    for label, col in cols.items():
        for s, l in groups:
            c = C(col, {"age": a_age, "sex": s, "layer": l}, {"age": b_age, "sex": s, "layer": l})
            out.append({"reading": label, "sex": s, "layer": LAYER[l], "contrast": f"P{a_age} - P{b_age}", **c})
    return pd.DataFrame(out)


def fmt(df):
    d = df.copy()
    for c in ["estimate", "lo", "hi"]:
        d[c] = d[c].map(lambda v: f"{v:.3f}")
    d["p"] = d["p"].map(lambda v: f"{v:.4f}")
    return d.to_string(index=False)


paper = dev_table[dev_table.Location != 4].rename(columns={"Age": "age", "Sex": "sex", "Layer": "layer", "Location": "location"})
paper["subject"] = paper.SubjectID
checks = [("EventFreq", "P11 - P21 0.605 (0.352 to 0.822), every group"),
          ("SynchronyFreq", "0.491 (0.315 to 0.658)"),
          ("ZF_R_allpair_mean", "F L2/3 0.153, F L4 0.096 (p = 0.017), M L2/3 0.076 (p = 0.096), M L4 -0.004 (p = 0.85)"),
          ("silhs_mean_beforeStat", "-0.188 to -0.215"),
          ("NCls", "0.575 (-0.443 to 1.763, p = 0.29)")]
for col, printed in checks:
    print(model_line(col, data=paper))
    t = age_table({col: col}, 11, 21) if False else None
    rows = []
    for s, l in GROUPS:
        c = C(col, {"age": 11, "sex": s, "layer": l}, {"age": 21, "sex": s, "layer": l}, data=paper)
        rows.append(f"{s} {LAYER[l]} {c['estimate']:.3f} ({c['lo']:.3f} to {c['hi']:.3f}, p = {c['p']:.3f})")
    print("   here P11 - P21:", "; ".join(rows))
    print("   repository output:", printed)

On the paper's own values the re-implemented test gives the contrasts the repository's outputs print,
to within a few hundredths, so it is used as test T1 below.

In [ ]:
EF = {"O6 a": "event_freq_active", "O6 b": "event_freq_all"}
SY = {"O2 Methods": "sync_methods", "O2 Step7": "sync_step7"}
SIL = {"O3 100 runs": "r100_silhouette", "O3 10 runs": "r10_silhouette"}
PEARSON = {"Pearson": "pearson"}
MI = {"MI": "mi"}


def CL(measure):
    return {f"O3 {R}, O4 {name}": f"r{R}_{p}_{measure}" for R in (100, 10) for name, p in (("95th", "methods"), ("code", "code"))}


def CLP(measure):
    out = {}
    for k, v in CL(measure).items():
        out[f"{k}, O5 a"] = v
        out[f"{k}, O5 b"] = v.replace(measure, measure + "_active")
    return out


def grade(n, total, partial_min):
    return "reproduced" if n == total else ("partially reproduced" if n >= partial_min else "not reproduced")


def by_reading(tab, rule, total=4, partial_min=2):
    out = {}
    for reading, g in tab.groupby("reading", sort=False):
        n = int(sum(rule(r) for _, r in g.iterrows()))
        out[reading] = (n, grade(n, total, partial_min))
    for k, (n, g) in out.items():
        print(f"  {k}: {n} of {total} groups hold -> {g}")
    return out


def ages_table(cols, groups=GROUPS):
    # all ten age contrasts, earlier minus later
    ages = LEV1["age"]
    out = []
    for label, col in cols.items():
        for s, l in groups:
            for i, a in enumerate(ages):
                for b in ages[i + 1:]:
                    c = C(col, {"age": a, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l})
                    out.append({"reading": label, "sex": s, "layer": LAYER[l], "a": a, "b": b, **c})
    return pd.DataFrame(out)


def emm(col, s, l):
    r = T(col)
    g = r["grid"]
    return {a: float(r["est"][(g.age.values == a) & (g.sex.values == s) & (g.layer.values == l)][0]) for a in LEV1["age"]}

## fig1d-event-frequency-decline

> Spontaneous event frequency during quiescence declines from P11 to P21 in both L2/3 and L4 neurons
> and in both sexes. (Figure 1D)

Criterion: P11 minus P21 positive with p < 0.05 in each of the four sex and layer groups (T1), under
the O6 readings.

In [ ]:
for col in EF.values():
    print(model_line(col))
tab = age_table(EF, 11, 21)
print(fmt(tab))
res_fig1d = by_reading(tab, lambda r: r.estimate > 0 and r.p < 0.05)

**Result: reproduced.** P11 minus P21 is 0.617 events/s (95% interval 0.418 to 0.813,
p < 0.0001) under O6 a and 0.621 under O6 b, with the main-effects model, so the same in all four
sex and layer groups. The repository's output was 0.605.

## fig1f-synchrony-decline

> The frequency of synchronous network events falls significantly by P21, in both layers and both
> sexes. (Figure 1E, 1F)

Criterion: P11 minus P21 positive with p < 0.05 in each sex and layer group, under the O2 readings.

In [ ]:
for col in SY.values():
    print(model_line(col))
tab = age_table(SY, 11, 21)
print(fmt(tab))
res_fig1f = by_reading(tab, lambda r: r.estimate > 0 and r.p < 0.05)

**Result: reproduced.** P11 minus P21 is 0.435 synchrony events/s (0.340 to 0.538,
p < 0.0001) with the Methods' threshold and 0.389 (0.232 to 0.538, p < 0.0001) with Step7's, the same
in every group (main-effects model).

## figs2f-pearson-decline-females

> The mean pairwise Pearson correlation falls from P11 to P21 in females but not in males. (Figure S2F)

Criterion: females, P11 minus P21 positive with p < 0.05 in both layers; males (a claim of absence),
p >= 0.05 in both layers.

In [ ]:
print(model_line("pearson"))
tab = age_table(PEARSON, 11, 21)
print(fmt(tab))
fem = tab[tab.sex == "F"]
mal = tab[tab.sex == "M"]
female_part = bool(((fem.estimate > 0) & (fem.p < 0.05)).all())
male_part = bool((mal.p >= 0.05).all())
print(f"female part holds: {female_part}; male part (p >= 0.05 in both layers) holds: {male_part}")

**Result: reproduced.** In females P11 minus P21 is 0.155 (p = 0.0002) in L2/3 and 0.096
(p = 0.010) in L4; in males 0.068 (p = 0.158) and -0.015 (p = 0.464). The selected model has the
three-way interaction, as in the repository.

## figs2f-pearson-l23-above-l4-p11

> At P11, L2/3 neurons have higher pairwise Pearson correlation than L4 neurons, in both sexes.

Criterion: T1 layer contrast at P11 within each sex; L2/3 above L4 with p < 0.05.

In [ ]:
rows = []
for s in "FM":
    c = C("pearson", {"age": 11, "sex": s, "layer": 4}, {"age": 11, "sex": s, "layer": 2})
    rows.append({"sex": s, "contrast": "L4 - L2/3 at P11", **c})
tab = pd.DataFrame(rows)
print(fmt(tab))
print("sexes with L2/3 above L4 at p < 0.05:", int(((tab.estimate < 0) & (tab.p < 0.05)).sum()), "of 2")

**Result: reproduced.** At P11, L4 minus L2/3 is -0.033 in females (p = 0.048, near the
threshold) and -0.052 in males (p < 0.0001).

## figs2g-mi-decline

> Mutual information between neuron pairs is significantly lower at P21 than at P11 in both sexes and
> both layers. (Figure S2G)

MI is that of `calculate_mi_matrix_trace.m` (256 bins) on the Z-scored traces of the window. The values here are about ten times those of the repository table's MI column and do not follow
them across datasets (data check).

In [ ]:
print(model_line("mi"))
tab = age_table(MI, 11, 21)
print(fmt(tab))
res = by_reading(tab, lambda r: r.estimate > 0 and r.p < 0.05)

**Result: blocked (methods underspecified).** With the repository's MI code (256 bins, Z-scored
traces of the window), MI is higher at P21 than at P11 (P11 minus P21 -0.160, p = 0.22), but these
MI values do not reproduce the paper's own per-dataset MI (r = 0.09 over 387 datasets, about ten
times larger) and follow the window length (r = -0.84), so the MI procedure here is not the
paper's.

## figs2g-mi-male-above-female-p21

> At P21, male neurons have significantly higher MI than female neurons, in both layers.

In [ ]:
rows = []
for l in (2, 4):
    c = C("mi", {"age": 21, "sex": "M", "layer": l}, {"age": 21, "sex": "F", "layer": l})
    rows.append({"layer": LAYER[l], "contrast": "M - F at P21", **c})
tab = pd.DataFrame(rows)
print(fmt(tab))
print("layers with M above F at p < 0.05:", int(((tab.estimate > 0) & (tab.p < 0.05)).sum()), "of 2")

**Result: blocked (methods underspecified).** Under the MI here, male minus female at P21 is
-0.071 (p = 0.33); the MI does not match the paper's (see figs2g-mi-decline).

## figs2g-mi-l4-below-l23

> L4 neurons have lower MI than L2/3 neurons at both P11 and P21, in both sexes.

In [ ]:
rows = []
for a in (11, 21):
    for s in "FM":
        c = C("mi", {"age": a, "sex": s, "layer": 4}, {"age": a, "sex": s, "layer": 2})
        rows.append({"age": a, "sex": s, "contrast": "L4 - L2/3", **c})
tab = pd.DataFrame(rows)
print(fmt(tab))
n = int(((tab.estimate < 0) & (tab.p < 0.05)).sum())
print(f"age and sex groups with L4 below L2/3 at p < 0.05: {n} of 4 -> {grade(n, 4, 2)}")

**Result: blocked (methods underspecified).** Under the MI here, L4 minus L2/3 is 0.085
(p = 0.096) in every age and sex group; the MI does not match the paper's (see figs2g-mi-decline).

## fig2e-subnetworks-present-same-number

> Neuronal subnetworks are already present at P11, and a similar number of subnetworks is found at
> P21. (Figure 2E)

Criterion: more than half of the P11 datasets have at least one subnetwork; and (a claim of absence)
the T1 P11 minus P21 contrast of subnetwork number has p >= 0.05 in every sex and layer group. Run
under the four O3 and O4 readings.

In [ ]:
rows = []
for label, col in CL("n_subnetworks").items():
    p11 = dev[dev.age == 11][col]
    rows.append({"reading": label, "P11 datasets": len(p11), "with >= 1 subnetwork": float((p11 >= 1).mean()),
                 "mean P11": p11.mean(), "mean P21": dev[dev.age == 21][col].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
for col in CL("n_subnetworks").values():
    print(model_line(col))
tab = age_table(CL("n_subnetworks"), 11, 21)
print(fmt(tab))
for reading, g in tab.groupby("reading", sort=False):
    print(f"  {reading}: groups with p >= 0.05: {int((g.p >= 0.05).sum())} of 4")
print("Per subject at P11 (mean subnetworks, 100 runs, 95th):",
      dev[dev.age == 11].groupby("subject").r100_methods_n_subnetworks.mean().round(1).to_dict())

**Result: reproduced, resting on the code's participation test.** Every P11 dataset (86 of 86)
has at least one subnetwork under all four readings. With the code's test (O4 b) P11 minus P21 has
p >= 0.05 in all four groups (10 runs: -0.075, p = 0.88; 100 runs: 0.41 and 0.35, p = 0.45 and 0.64).
With the Methods' 95th percentile, P21 has more subnetworks in some groups (100 runs: 2 of 4 groups
with p < 0.05; 10 runs: 3 of 4), which would not reproduce the equal number.

## fig2d-silhouette-increase

> The silhouette value of event clusters increases substantially from P11 to P21 in both layers and
> both sexes. (Figure 2D)

The silhouette value is the mean over clusters of each cluster's median silhouette of the best
k-means run, before the cluster significance test (claims file). Criterion: P21 above P11 with
p < 0.05 in all four groups, under the O3 readings.

In [ ]:
for col in SIL.values():
    print(model_line(col))
tab = age_table(SIL, 11, 21)
print(fmt(tab))
res = by_reading(tab, lambda r: r.estimate < 0 and r.p < 0.05)
print("datasets without a silhouette value (no valid run):", int(dev.r100_silhouette.isna().sum()), int(dev.r10_silhouette.isna().sum()))

**Result: reproduced.** P21 exceeds P11 by 0.206 (0.165 to 0.258, p < 0.0001) with 100
k-means runs and by 0.185 (0.149 to 0.230, p < 0.0001) with 10, the same in all four groups.

## figs3c-tcc-inside-above-outside

> At P11 and P21 the tetrachoric correlation of neuron pairs within the same subnetwork is
> consistently higher than that of pairs from different subnetworks. (Figure S3C)

Criterion: on datasets with at least two subnetworks, mean inside above mean outside in each of the
eight age, sex and layer groups, and a paired Wilcoxon signed-rank test over datasets with p < 0.05
at each age.

In [ ]:
for label in CL("tcc_in"):
    cin, cout = CL("tcc_in")[label], CL("tcc_out")[label]
    nsub = CL("n_subnetworks")[label]
    d = dev[(dev.age.isin([11, 21])) & (dev[nsub] >= 2)].dropna(subset=[cin, cout])
    g = d.groupby(["age", "sex", "layer"]).agg(n=(cin, "size"), inside=(cin, "mean"), outside=(cout, "mean"))
    g["inside_above"] = g.inside > g.outside
    print(f"{label}: datasets {len(d)}; fraction with inside above outside {np.mean(d[cin] > d[cout]):.3f}")
    print(g.round(3).to_string())
    for a in (11, 21):
        x = d[d.age == a]
        w = stats.wilcoxon(x[cin], x[cout])
        print(f"   P{a}: n = {len(x)}, median difference {np.median(x[cin]-x[cout]):.3f}, Wilcoxon p = {w.pvalue:.2e}")
    print(f"   groups with inside above outside: {int(g.inside_above.sum())} of {len(g)}")

**Result: reproduced.** Under all four readings the mean tCC inside subnetworks exceeds that
between subnetworks in all eight age, sex and layer groups, in 99.2 to 100% of datasets, and the
Wilcoxon test gives p below 1e-12 at P11 (n = 85) and at P21 (n = 45 to 50).

## fig2f-tcc-increase-l23

> Within-subnetwork tCC is significantly higher at P21 than P11 in L2/3 in both sexes, while L4 shows a
> similar trend that is not significant (p = 0.051). (Figure 2F)

Criterion: L2/3 P21 minus P11 positive with p < 0.05 in both sexes and the L4 contrast positive with
p >= 0.05; partially reproduced when L2/3 holds and L4 is positive with p < 0.05.

In [ ]:
for col in CL("tcc_in").values():
    print(model_line(col))
tab = age_table(CL("tcc_in"), 21, 11)
print(fmt(tab))
for reading, g in tab.groupby("reading", sort=False):
    l23 = g[g.layer == "L2/3"]
    l4 = g[g.layer == "L4"]
    l23_ok = bool(((l23.estimate > 0) & (l23.p < 0.05)).all())
    l4_trend = bool(((l4.estimate > 0) & (l4.p >= 0.05)).all())
    l4_sig = bool(((l4.estimate > 0) & (l4.p < 0.05)).all())
    out = "reproduced" if l23_ok and l4_trend else ("partially reproduced" if l23_ok and l4_sig else "not reproduced")
    print(f"  {reading}: L2/3 holds {l23_ok}, L4 positive and p >= 0.05 {l4_trend}, L4 positive and p < 0.05 {l4_sig} -> {out}")

**Result: not reproduced, resting on the code's readings.** With 10 runs and the code's test,
P21 minus P11 of within-subnetwork tCC is 0.056 (p = 0.106) in every group, so the L2/3 increase is
not significant. With 100 runs and the code's test it is 0.080 (p = 0.007) in every group, L4
included (partially reproduced under that reading); with the Methods' test it is negative (-0.009
and -0.013).

## figs3d-neuron-number-not-related and figs3e-time-interval-not-related

> The number of subnetworks found is barely related to the number of neurons in the analysis
> (Spearman rho = -0.063). The length of the analysed time interval has little effect on the number of
> subnetworks (Spearman rho = 0.233). (Figure S3D, S3E)

Criterion: over the P11 and P21 datasets pooled, Spearman correlation of subnetwork number with the
number of ROIs and with the window length; reproduced when rho is within 0.08 of -0.063 and of 0.233.

In [ ]:
d = dev[dev.age.isin([11, 21])]
rows = []
for label, col in CL("n_subnetworks").items():
    r1 = stats.spearmanr(d[col], d.n_rois)
    r2 = stats.spearmanr(d[col], d.seconds)
    rows.append({"reading": label, "n": len(d), "rho_rois": r1.statistic, "p_rois": r1.pvalue,
                 "within_tol_rois": abs(r1.statistic + 0.063) <= 0.08,
                 "rho_interval": r2.statistic, "p_interval": r2.pvalue, "within_tol_interval": abs(r2.statistic - 0.233) <= 0.08})
print(pd.DataFrame(rows).round(4).to_string(index=False))
r3 = stats.spearmanr(d.NCls, d.total_cell_number, nan_policy="omit")
r4 = stats.spearmanr(d.NCls, d.Time_interval, nan_policy="omit")
print(f"Same datasets, the table's subnetwork numbers: rho with ROIs {r3.statistic:.3f}, with interval {r4.statistic:.3f}")

**Result: reproduced, resting on the code's participation test.** Over 137 P11 and P21
datasets, rho with the number of ROIs is -0.119 (10 runs) and -0.090 (100 runs) with the code's
test, within 0.08 of -0.063; with the Methods' test it is -0.313 and -0.253, outside it. The table's
own subnetwork numbers give -0.045 on the same datasets.

In [ ]:
print(pd.DataFrame(rows)[["reading", "n", "rho_interval", "p_interval", "within_tol_interval"]].round(4).to_string(index=False))

**Result: reproduced, resting on the code's participation test.** Rho with the window length is
0.237 (10 runs) and 0.234 (100 runs) with the code's test, and 0.177 with 100 runs and the Methods'
test, all within 0.08 of 0.233; with 10 runs and the Methods' test it is 0.094.

## fig3a-single-cluster-decline, fig3b-non-cluster-increase, fig3c-multiple-cluster-unchanged

> The percentage of neurons participating in a single event cluster decreases from P11 to P21, in both
> layers and sexes; the percentage not participating in any rises; the percentage in multiple
> clusters is similar at P11 and P21. (Figure 3A to 3C)

Run under the O3, O4 and O5 readings (eight per measure).

In [ ]:
tab_single = age_table(CLP("pct_single"), 11, 21)
print(fmt(tab_single))
res_single = by_reading(tab_single, lambda r: r.estimate > 0 and r.p < 0.05)

**Result: reproduced.** P11 minus P21 of single-cluster neurons is positive with p < 0.05 in all
four groups under all eight readings; with 10 runs, the code's test and all ROIs (O5 a) it is 13.2
to 37.5 percentage points (p from 0.035 to below 0.0001).

In [ ]:
tab_non = age_table(CLP("pct_non"), 11, 21)
print(fmt(tab_non))
res_non = by_reading(tab_non, lambda r: r.estimate < 0 and r.p < 0.05)

**Result: reproduced, resting on the code's participation test.** With the code's test P21
exceeds P11 by 19.5 points (10 runs, O5 a; p < 0.0001) and by 17.3 to 21.1 under the other code
readings, in every group. With the Methods' test the rise is smaller: 5.3 points with 10 runs and
O5 a (p = 0.024), and not significant under O5 b (3.3 with 10 runs, p = 0.137; 4.9 with 100 runs,
p = 0.053).

In [ ]:
tab_mult = age_table(CLP("pct_multiple"), 11, 21)
print(fmt(tab_mult))
for reading, g in tab_mult.groupby("reading", sort=False):
    print(f"  {reading}: groups with p < 0.05: {int((g.p < 0.05).sum())} of 4; estimates {g.estimate.round(1).tolist()}")

**Result: reproduced, resting on the code's participation test.** With 10 runs and the code's
test, P11 minus P21 of multiple-cluster neurons is -7.7 points (p = 0.149) in every group; with 100
runs -3.0 (p = 0.61). With 10 runs and the Methods' test it is -14.6 (p = 0.046), which would not
reproduce.

## fig3ef-subnetwork-size-unchanged

> The physical size of subnetworks (pairwise distances between member neurons and distances to the
> subnetwork centroid) does not differ between P11 and P21 within a layer. (Figure 3E, 3F)

A claim of absence: every T1 P11 minus P21 contrast of each measure in each layer and sex has
p >= 0.05. Distances are in pixels of the ROI centres.

In [ ]:
tab_pw = age_table(CL("pw_neuron"), 11, 21)
tab_nc = age_table(CL("neuron_centroid"), 11, 21)
print("Neuron to neuron:\n" + fmt(tab_pw))
print("Neuron to centroid:\n" + fmt(tab_nc))
for reading in CL("pw_neuron"):
    a = tab_pw[tab_pw.reading == reading]
    b = tab_nc[tab_nc.reading == reading]
    print(f"  {reading}: contrasts with p < 0.05: neuron-neuron {int((a.p < 0.05).sum())} of 4, "
          f"neuron-centroid {int((b.p < 0.05).sum())} of 4")

**Result: reproduced, resting on the code's participation test.** With 10 runs and the code's
test, P11 minus P21 is -2.7 pixels (p = 0.62) for neuron to neuron and 2.9 (p = 0.49) for neuron to
centroid distances, in every group; with 100 runs -0.8 (p = 0.85) and 2.4 (p = 0.54). With the
Methods' test the neuron to neuron distance is larger at P21 (-12.4 and -15.0 pixels, p < 0.01).

## fig3g-centroid-distances-decrease

> Pairwise distances between subnetwork centroids fall significantly from P11 to P21 in both layers and
> sexes. (Figure 3D, 3G)

On datasets with at least two subnetworks (the measure is undefined otherwise).

In [ ]:
for col in CL("centroid_centroid").values():
    print(model_line(col))
tab = age_table(CL("centroid_centroid"), 11, 21)
print(fmt(tab))
res = by_reading(tab, lambda r: r.estimate > 0 and r.p < 0.05)

**Result: reproduced.** P11 minus P21 of centroid to centroid distances is positive with
p < 0.0001 in all four groups under all four readings: 57.2 pixels with 10 runs and the code's test,
54.0 to 80.3 under the others.

## fig4a-event-frequency-trajectory

> Across P11 to P21 event frequency declines, with the largest mean difference between P11 and P18.
> (Figure 4A)

Criterion: of the ten age contrasts, P11 minus P18 has the largest positive estimate and p < 0.05.
Judged in every sex and layer group (identical when the selected model has no age interaction).

In [ ]:
tab = ages_table(EF)
for reading, g in tab.groupby("reading", sort=False):
    print(reading)
    ok_all = True
    for (s, l), h in g.groupby(["sex", "layer"]):
        top = h.loc[h.estimate.idxmax()]
        p1118 = h[(h.a == 11) & (h.b == 18)].iloc[0]
        p1121 = h[(h.a == 11) & (h.b == 21)].iloc[0]
        ok = top.a == 11 and top.b == 18 and top.p < 0.05
        ok_all &= ok
        print(f"  {s} {l}: largest P{top.a} - P{top.b} {top.estimate:.3f} (p = {top.p:.4f}); "
              f"P11 - P18 {p1118.estimate:.3f}, P11 - P21 {p1121.estimate:.3f} -> {ok}")
    print(f"  holds in every group: {ok_all}")

**Result: not reproduced.** The largest of the ten age contrasts is P11 minus P21 (0.617
events/s), not P11 minus P18 (0.579), under both O6 readings; both are significant (p < 0.0001). The
repository's output had them in the other order (0.635 and 0.605).

## fig4b-synchrony-gradual-decline

> Synchrony frequency declines gradually from P11 to P21. (Figure 4B)

Criterion: the estimated marginal means fall at every step from P11 to P21 except at most one step
that does not rise significantly, and P11 minus P21 is positive with p < 0.05.

In [ ]:
ages = LEV1["age"]
for label, col in SY.items():
    print(model_line(col))
    ok_all = True
    for s, l in GROUPS:
        m = emm(col, s, l)
        steps = [C(col, {"age": a, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l}) for a, b in zip(ages[:-1], ages[1:])]
        falls = [st["estimate"] > 0 for st in steps]
        sig_rise = [st["estimate"] < 0 and st["p"] < 0.05 for st in steps]
        nonfall = [i for i, f in enumerate(falls) if not f]
        c = C(col, {"age": 11, "sex": s, "layer": l}, {"age": 21, "sex": s, "layer": l})
        ok = len(nonfall) <= 1 and not any(sig_rise) and c["estimate"] > 0 and c["p"] < 0.05
        ok_all &= ok
        print(f"  {label} {s} {LAYER[l]}: means {[round(v, 3) for v in m.values()]}; steps (earlier - later) "
              f"{[round(st['estimate'], 3) for st in steps]} p {[round(st['p'], 3) for st in steps]}; "
              f"P11 - P21 {c['estimate']:.3f} p {c['p']:.4f} -> {ok}")
    print(f"  {label}: holds in every group: {ok_all}")

**Result: reproduced.** The marginal means fall at every step from P11 to P21 under both
readings (Methods: steps 0.065, 0.114, 0.187, 0.069; Step7: 0.015, 0.031, 0.254, 0.088), none rises,
and P11 minus P21 is 0.435 and 0.389 (p < 0.0001).

## figs4a-pearson-trajectory

> In females the Pearson correlation decreases gradually from P11 to P21 in L2/3 and from P11 to P18 in
> L4; in males it decreases from P11 to P15 and then rises transiently at P18. (Figure S4A)

In [ ]:
print(model_line("pearson"))
tab = ages_table(PEARSON)
def get(s, l, a, b):
    return tab[(tab.sex == s) & (tab.layer == l) & (tab.a == a) & (tab.b == b)].iloc[0]
f23 = get("F", "L2/3", 11, 21)
part1 = f23.estimate > 0 and f23.p < 0.05
f4 = tab[(tab.sex == "F") & (tab.layer == "L4") & (tab.a == 11)]
top = f4.loc[f4.estimate.idxmax()]
f418 = get("F", "L4", 11, 18)
part2 = f418.estimate > 0 and f418.p < 0.05 and top.b == 18
m15 = [get("M", l, 11, 15) for l in ("L2/3", "L4")]
m1518 = [get("M", l, 15, 18) for l in ("L2/3", "L4")]
part3 = any(r.estimate > 0 and r.p < 0.05 for r in m15) and any(r.estimate < 0 and r.p < 0.05 for r in m1518)
print(f"female L2/3 P11 - P21 {f23.estimate:.3f} (p = {f23.p:.4f}) -> {part1}")
print(f"female L4 contrasts from P11: " + ", ".join(f"P{r.b} {r.estimate:.3f} (p = {r.p:.4f})" for _, r in f4.iterrows()) + f" -> {part2}")
print("male P11 - P15: " + ", ".join(f"{r.layer} {r.estimate:.3f} (p = {r.p:.4f})" for r in m15))
print("male P15 - P18: " + ", ".join(f"{r.layer} {r.estimate:.3f} (p = {r.p:.4f})" for r in m1518) + f" -> {part3}")
n = int(part1) + int(part2) + int(part3)
print(f"parts holding: {n} of 3 -> {grade(n, 3, 1)}")

**Result: reproduced.** Female L2/3 P11 minus P21 is 0.155 (p = 0.0002); in female L4 the
largest contrast from P11 is to P18 (0.102, p < 0.0001); in males P11 minus P15 is 0.072 and 0.070
(p = 0.0008 and 0.008) and P15 minus P18 is -0.047 and -0.088 (p = 0.024 and below 0.0001).

## figs4b-mi-trajectory

> MI declines over time in females from P11 to P21 and in males from P11 to P15, and L2/3 neurons have
> higher MI than L4 neurons at every age. (Figure S4B)

In [ ]:
dec = []
for s, b in (("F", 21), ("M", 15)):
    for l in (2, 4):
        c = C("mi", {"age": 11, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l})
        dec.append({"sex": s, "layer": LAYER[l], "contrast": f"P11 - P{b}", **c})
dec = pd.DataFrame(dec)
print(fmt(dec))
lay = []
for a in LEV1["age"]:
    for s in "FM":
        c = C("mi", {"age": a, "sex": s, "layer": 2}, {"age": a, "sex": s, "layer": 4})
        lay.append({"age": a, "sex": s, "contrast": "L2/3 - L4", **c})
lay = pd.DataFrame(lay)
print(fmt(lay))
p1 = bool(((dec.estimate > 0) & (dec.p < 0.05)).all())
p2 = bool(((lay.estimate > 0) & (lay.p < 0.05)).all())
print(f"decline part {p1}; layer part {p2} ({int(((lay.estimate > 0) & (lay.p < 0.05)).sum())} of 10 groups)")

**Result: blocked (methods underspecified).** Under the MI here, female P11 minus P21 is -0.160
(p = 0.22), male P11 minus P15 -0.477 (p < 0.0001), and L2/3 minus L4 -0.085 (p = 0.096) at every
age; the MI does not match the paper's (see figs2g-mi-decline).

## figs4b-mi-sex-p21

> Females have higher MI than males at P21.

Criterion: female above male at P21 with p < 0.05 in at least one layer. The claims file notes that
this sentence contradicts the Figure S2G sentence on the same model.

In [ ]:
rows = []
for l in (2, 4):
    c = C("mi", {"age": 21, "sex": "F", "layer": l}, {"age": 21, "sex": "M", "layer": l})
    rows.append({"layer": LAYER[l], "contrast": "F - M at P21", **c})
tab = pd.DataFrame(rows)
print(fmt(tab))
print("layers with F above M at p < 0.05:", int(((tab.estimate > 0) & (tab.p < 0.05)).sum()))

**Result: blocked (methods underspecified).** Under the MI here, female minus male at P21 is
0.071 (p = 0.33) in both layers; the MI does not match the paper's (see figs2g-mi-decline).

## fig4c-subnetwork-number-transient-p15

> The number of subnetworks shows a transient, modest increase at P15. (Figure 4C)

Criterion: P15 above P11 with p < 0.05 and P15 above P21 with p < 0.05; P13 against P11 is reported.

In [ ]:
for label, col in CL("n_subnetworks").items():
    out = []
    for s, l in GROUPS:
        a = C(col, {"age": 15, "sex": s, "layer": l}, {"age": 11, "sex": s, "layer": l})
        b = C(col, {"age": 15, "sex": s, "layer": l}, {"age": 21, "sex": s, "layer": l})
        c = C(col, {"age": 13, "sex": s, "layer": l}, {"age": 11, "sex": s, "layer": l})
        ok = a["estimate"] > 0 and a["p"] < 0.05 and b["estimate"] > 0 and b["p"] < 0.05
        out.append(ok)
        print(f"  {label} {s} {LAYER[l]}: P15 - P11 {a['estimate']:.2f} (p = {a['p']:.4f}); P15 - P21 {b['estimate']:.2f} "
              f"(p = {b['p']:.4f}); P13 - P11 {c['estimate']:.2f} (p = {c['p']:.4f}) -> {ok}")
    print(f"  {label}: groups holding {sum(out)} of 4; means by age",
          dev.groupby("age")[col].mean().round(2).to_dict())

**Result: reproduced, resting on the code's readings.** With 10 runs and the code's test, P15
exceeds P11 by 0.98 (p = 0.020) and P21 by 0.90 (p = 0.002) in all four groups, and P13 also exceeds
P11 (0.98, p = 0.001). Under the other readings it holds in 0 to 2 groups.

## fig4d-silhouette-gradual-increase

> The silhouette value increases gradually from P11 to P21 regardless of sex and layer, and at P15
> males have transiently higher silhouette values than females in both layers. (Figure 4D)

In [ ]:
for label, col in SIL.items():
    print(model_line(col))
    inc_ok = True
    for s, l in GROUPS:
        c = C(col, {"age": 21, "sex": s, "layer": l}, {"age": 11, "sex": s, "layer": l})
        steps = [C(col, {"age": b, "sex": s, "layer": l}, {"age": a, "sex": s, "layer": l}) for a, b in zip(ages[:-1], ages[1:])]
        sig_fall = any(st["estimate"] < 0 and st["p"] < 0.05 for st in steps)
        ok = c["estimate"] > 0 and c["p"] < 0.05 and not sig_fall
        inc_ok &= ok
        print(f"  {s} {LAYER[l]}: P21 - P11 {c['estimate']:.3f} (p = {c['p']:.4f}); steps later - earlier "
              f"{[round(st['estimate'], 3) for st in steps]} p {[round(st['p'], 3) for st in steps]} -> {ok}")
    sex_rows = []
    for a in (11, 15, 21):
        for l in (2, 4):
            c = C(col, {"age": a, "sex": "M", "layer": l}, {"age": a, "sex": "F", "layer": l})
            sex_rows.append({"age": a, "layer": LAYER[l], **c})
    sx = pd.DataFrame(sex_rows)
    p15 = sx[sx.age == 15]
    other = sx[sx.age != 15]
    sex_ok = bool(((p15.estimate > 0) & (p15.p < 0.05)).all()) and not bool(((other.estimate > 0) & (other.p < 0.05)).any())
    print("  M - F:\n" + fmt(sx))
    print(f"  {label}: increase part {inc_ok}; sex part {sex_ok}")

**Result: partially reproduced.** The increase holds under both readings: P21 minus P11 is
0.206 and 0.185 (p < 0.0001) and every step rises (p from below 0.0001 to 0.029). The sex part does
not: male minus female at P15 is 0.006 (p = 0.81) with 100 runs and 0.005 (p = 0.83) with 10, as at
every age (main-effects model).

## fig4e-tcc-maximum-earlier-l23

> Within-subnetwork tCC reaches its maximum earlier in L2/3 than in L4, with the largest mean difference
> from P11 at P15 in L2/3 and at P18 in L4, in both sexes. (Figure 4E)

In [ ]:
for label, col in CL("tcc_in").items():
    oks = []
    for s, l in GROUPS:
        cs = {b: C(col, {"age": b, "sex": s, "layer": l}, {"age": 11, "sex": s, "layer": l}) for b in (13, 15, 18, 21)}
        top = max(cs, key=lambda b: cs[b]["estimate"])
        want = 15 if l == 2 else 18
        ok = top == want and cs[top]["p"] < 0.05
        oks.append(ok)
        print(f"  {label} {s} {LAYER[l]}: later - P11 " + ", ".join(f"P{b} {cs[b]['estimate']:.3f} (p = {cs[b]['p']:.3f})" for b in cs) + f" -> largest P{top}, {ok}")
    print(f"  {label}: groups holding {sum(oks)} of 4")

**Result: not reproduced.** Under no reading is the largest within-subnetwork tCC increase
from P11 at P15 in L2/3: with 10 runs and the code's test it is at P18 (0.071, p = 0.001) in both
layers, with 100 runs and the code's test at P21 (0.080), and with the Methods' test at P13 (0.030
and 0.037, not significant).

## fig4e-tcc-l4-below-l23-p15

> At P15, L4 neurons within the same subnetwork have significantly lower tCC than L2/3 neurons.

In [ ]:
for label, col in CL("tcc_in").items():
    rows = []
    for s in "FM":
        c = C(col, {"age": 15, "sex": s, "layer": 4}, {"age": 15, "sex": s, "layer": 2})
        rows.append(f"{s} {c['estimate']:.3f} ({c['lo']:.3f} to {c['hi']:.3f}, p = {c['p']:.4f})")
    print(f"  {label}: L4 - L2/3 at P15: " + "; ".join(rows))

**Result: reproduced, resting on the code's readings.** With 10 runs and the code's test, L4
minus L2/3 at P15 is -0.026 (-0.054 to -0.002, p = 0.028) in both sexes; with the Methods' test
-0.041 and -0.048 (p = 0.003 and 0.002). With 100 runs and the code's test it is -0.016
(p = 0.17).

## fig4fgh-configuration-trajectory

> Single-cluster neurons decrease gradually from P11 to P18, non-cluster neurons increase from P11 to
> P21, and multiple-cluster neurons stay constant throughout. (Figure 4F to 4H)

In [ ]:
summary_rows = []
for label in CLP("pct_single"):
    cs, cn, cm = CLP("pct_single")[label], CLP("pct_non")[label], CLP("pct_multiple")[label]
    single_ok, non_ok, mult_ok = True, True, True
    nsig_mult = 0
    for s, l in GROUPS:
        fr = {b: C(cs, {"age": 11, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l}) for b in (13, 15, 18, 21)}
        top = max(fr, key=lambda b: fr[b]["estimate"])
        single_ok &= top == 18 and fr[18]["p"] < 0.05 and fr[18]["estimate"] > 0
        n = C(cn, {"age": 21, "sex": s, "layer": l}, {"age": 11, "sex": s, "layer": l})
        non_ok &= n["estimate"] > 0 and n["p"] < 0.05
        for i, a in enumerate(ages):
            for b in ages[i + 1:]:
                m = C(cm, {"age": a, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l})
                nsig_mult += m["p"] < 0.05
    mult_ok = nsig_mult == 0
    k = int(single_ok) + int(non_ok) + int(mult_ok)
    summary_rows.append({"reading": label, "single": single_ok, "non": non_ok, "multiple": mult_ok,
                         "multiple contrasts p<0.05 (of 40)": nsig_mult, "outcome": grade(k, 3, 2)})
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\nMain reading, single-cluster contrasts from P11 (F L2/3):",
      {b: round(C("r100_methods_pct_single", {"age": 11, "sex": "F", "layer": 2}, {"age": b, "sex": "F", "layer": 2})["estimate"], 2) for b in (13, 15, 18, 21)})
print("Main reading, mean percentages by age:")
print(dev.groupby("age")[["r100_methods_pct_single", "r100_methods_pct_non", "r100_methods_pct_multiple"]].mean().round(1).to_string())

**Result: not reproduced, resting on the code's readings.** With 10 runs, the code's test and
O5 a, non-cluster neurons rise from P11 to P21, but the largest single-cluster contrast from P11 is
not to P18 in every group, and 12 of the 40 age contrasts of multiple-cluster neurons have p < 0.05.
Under the Methods' test with 10 runs and O5 a, two parts hold (partially reproduced); under every
other reading one part holds. The repository's own output also had a multiple-cluster contrast with
p = 0.029.

## figs4-distance-trajectory

> The largest mean differences in subnetwork size (neuron to neuron and neuron to centroid distances)
> occur between P13 and P21, and centroid to centroid distances decrease gradually, most from P11 to
> P21 in L2/3 and from P11 to P18 in L4. (Figure S4C to S4E)

Criterion: of the ten age contrasts, P13 against P21 has the largest absolute estimate for both size
measures, and the largest centroid distance reduction (earlier minus later) is P11 to P21 in L2/3 and
P11 to P18 in L4. Judged in every sex and layer group.

In [ ]:
for label in CL("pw_neuron"):
    size_ok, cent_ok = True, True
    lines = []
    for s, l in GROUPS:
        for meas in ("pw_neuron", "neuron_centroid"):
            col = CL(meas)[label]
            cs = [(a, b, C(col, {"age": a, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l})) for i, a in enumerate(ages) for b in ages[i + 1:]]
            a, b, c = max(cs, key=lambda x: abs(x[2]["estimate"]))
            size_ok &= (a, b) == (13, 21)
            lines.append(f"{s} {LAYER[l]} {meas}: largest |P{a} - P{b}| = {c['estimate']:.1f} (p = {c['p']:.3f})")
        col = CL("centroid_centroid")[label]
        cs = [(a, b, C(col, {"age": a, "sex": s, "layer": l}, {"age": b, "sex": s, "layer": l})) for i, a in enumerate(ages) for b in ages[i + 1:]]
        a, b, c = max(cs, key=lambda x: x[2]["estimate"])
        want = (11, 21) if l == 2 else (11, 18)
        cent_ok &= (a, b) == want
        lines.append(f"{s} {LAYER[l]} centroid: largest reduction P{a} - P{b} = {c['estimate']:.1f} (p = {c['p']:.3f})")
    print(label)
    print("  " + "\n  ".join(lines))
    k = int(size_ok) + int(cent_ok)
    print(f"  size part {size_ok}; centroid part {cent_ok} -> {grade(k, 2, 1)}")

**Result: not reproduced, resting on the code's readings.** With 10 runs and the code's test,
the largest size contrasts are P13 minus P18 (neuron to neuron, -16.5 pixels) and P11 minus P13
(neuron to centroid, 10.9), and the largest centroid reduction is P11 minus P21 (57.2) in L4 as in
L2/3. With the Methods' test both size measures peak at P13 against P21 (partially reproduced), and
the centroid part fails under every reading.

## The DREADD experiment (test T2)

T2 has fixed effects CNO (before or after), sex and layer, selected and bootstrapped as T1. The
DREADD datasets are the P15 files of injected mice before CNO and the files after CNO, each in the
window of the repository's DREADD table.

In [ ]:
def cno_table(cols, groups=GROUPS):
    out = []
    for label, col in cols.items():
        for s, l in groups:
            c = C(col, {"cno": "before", "sex": s, "layer": l}, {"cno": "after", "sex": s, "layer": l}, kind="dre")
            out.append({"reading": label, "sex": s, "layer": LAYER[l], "contrast": "before - after", **c})
    return pd.DataFrame(out)


def model_line2(col):
    return model_line(col, kind="dre")

## fig5cd-cno-reduces-events-synchrony

> Acutely enhancing GABAergic activity with CNO at P15 to P16 reduces both the frequency of calcium
> events and of synchronous events, in both layers and sexes. (Figure 5C, 5D)

In [ ]:
for col in list(EF.values()) + list(SY.values()):
    print(model_line2(col))
te = cno_table(EF)
ts = cno_table(SY)
print(fmt(te))
print(fmt(ts))
for re_ in EF:
    for rs in SY:
        a = te[te.reading == re_]
        b = ts[ts.reading == rs]
        ea = bool(((a.estimate > 0) & (a.p < 0.05)).all())
        sb = bool(((b.estimate > 0) & (b.p < 0.05)).all())
        print(f"  {re_} with {rs}: events hold {ea}, synchrony hold {sb} -> {grade(int(ea) + int(sb), 2, 1)}")

**Result: reproduced.** Before minus after CNO, event frequency falls by 0.337 events/s
(p = 0.002) and synchrony by 0.214 (Methods, p = 0.025) or 0.365 (Step7, p = 0.0007), in every group
(main-effects models), over 56 datasets of 7 subjects.

## figs6ab-cno-correlation-unchanged

> CNO does not detectably change the Pearson correlation or the MI of all neuron pairs. (Figure S6A)

A claim of absence on the T2 before against after contrast of each measure.

In [ ]:
for col in ("pearson", "mi"):
    print(model_line2(col))
tab = cno_table({"Pearson": "pearson", "MI": "mi"})
print(fmt(tab))
for reading, g in tab.groupby("reading", sort=False):
    print(f"  {reading}: groups with p < 0.05: {int((g.p < 0.05).sum())} of 4")

**Result: not reproduced.** The Pearson correlation is higher after CNO by 0.045 (-0.074 to
-0.015 before minus after, p = 0.004), as the repository's own output found (0.030, p = 0.040). MI
here also changes (0.314, p < 0.0001), but that MI does not match the paper's.

## fig5e-cno-silhouette-increase

> CNO robustly increases the silhouette value of event clusters in both layers, independent of sex.
> (Figure 5E)

In [ ]:
for col in SIL.values():
    print(model_line2(col))
tab = cno_table(SIL)
print(fmt(tab))
res = by_reading(tab, lambda r: r.estimate < 0 and r.p < 0.05)

**Result: reproduced.** The silhouette value is higher after CNO by 0.092 (100 runs,
p = 0.0007) and 0.077 (10 runs, p = 0.004), in every group.

## figs6c-cno-subnetwork-number-unchanged

> CNO leaves the number of subnetworks unchanged. (Figure S6B)

In [ ]:
for col in CL("n_subnetworks").values():
    print(model_line2(col))
tab = cno_table(CL("n_subnetworks"))
print(fmt(tab))
for reading, g in tab.groupby("reading", sort=False):
    print(f"  {reading}: groups with p < 0.05: {int((g.p < 0.05).sum())} of 4")

**Result: not reproduced.** In males the number of subnetworks is higher after CNO under every
reading (before minus after -1.9 to -3.0, p < 0.0001; two male subjects); in females it does not
change significantly except under 10 runs and the Methods' test (1.14, p = 0.028).

## figs6d-cno-tcc-females

> CNO increases within-subnetwork tCC, an effect evident in females in both L2/3 and L4. (Figure S6C)

In [ ]:
for col in CL("tcc_in").values():
    print(model_line2(col))
tab = cno_table(CL("tcc_in"))
print(fmt(tab))
for reading, g in tab.groupby("reading", sort=False):
    f = g[g.sex == "F"]
    print(f"  {reading}: females with after above before at p < 0.05: {int(((f.estimate < 0) & (f.p < 0.05)).sum())} of 2")

**Result: reproduced.** Within-subnetwork tCC is higher after CNO by 0.116 to 0.137
(p <= 0.0007) under every reading, in females of both layers; the selected models have no CNO by sex
term, so males show the same estimate.

## fig5fgh-cno-configuration

> CNO decreases the percentage of single-cluster neurons and increases the percentage of
> multiple-cluster neurons, while the percentage of non-cluster neurons does not change significantly.
> (Figure 5F to 5H)

Judged in every sex and layer group.

In [ ]:
ts, tm, tn = cno_table(CLP("pct_single")), cno_table(CLP("pct_multiple")), cno_table(CLP("pct_non"))
print("single:\n" + fmt(ts))
print("multiple:\n" + fmt(tm))
print("non:\n" + fmt(tn))
for reading in CLP("pct_single"):
    a, b, c = ts[ts.reading == reading], tm[tm.reading == reading], tn[tn.reading == reading]
    s_ok = bool(((a.estimate > 0) & (a.p < 0.05)).all())
    m_ok = bool(((b.estimate < 0) & (b.p < 0.05)).all())
    n_ok = bool((c.p >= 0.05).all())
    k = int(s_ok) + int(m_ok) + int(n_ok)
    print(f"  {reading}: single {s_ok}, multiple {m_ok}, non unchanged {n_ok} -> {grade(k, 3, 2)}")

**Result: not reproduced, resting on the code's readings.** With 10 runs, the code's test and
O5 a, single-cluster neurons do not fall (3.1 points, p = 0.54), multiple-cluster neurons rise (15.5,
p = 0.003) and non-cluster neurons fall (12.3, p = 0.014). With 100 runs and the Methods' test,
single and multiple hold and the non-cluster change is significant (partially reproduced).

## figs6efg-cno-topography-unchanged

> CNO has no detectable effect on distances between neurons of a subnetwork, distances of neurons to
> their centroid, or distances between subnetwork centroids. (Figure S6D to S6F)

In [ ]:
for label in CL("pw_neuron"):
    sig = {}
    for meas in ("pw_neuron", "neuron_centroid", "centroid_centroid"):
        t = cno_table({label: CL(meas)[label]})
        sig[meas] = int((t.p < 0.05).sum())
        print(f"  {label} {meas}: " + "; ".join(f"{r.sex} {r.layer} {r.estimate:.1f} (p = {r.p:.3f})" for _, r in t.iterrows()))
    print(f"  {label}: contrasts with p < 0.05 {sig}")

**Result: not reproduced, resting on the code's readings.** With 10 runs and the code's test,
the centroid to centroid distance changes in males (66.2 pixels, p < 0.0001); with 100 runs and the
code's test no distance changes (p from 0.54 to 0.76); with the Methods' test the neuron to neuron
distance changes (10.5 and 11.8 pixels, p <= 0.004).

## figs7a-events-clustered

> k-means on the covariance matrix groups about 25% of events into clusters, about 1% of events in
> multiple clusters, and about 30% at P11.

Events are the frames of the window; a frame is in a cluster when its count of active subnetwork
neurons exceeds the shuffle threshold of `Modol_script.m` (PCl). Mean over the developmental datasets.

In [ ]:
rows = []
for label in CL("pct_frames_single"):
    s1, m1 = CL("pct_frames_single")[label], CL("pct_frames_multiple")[label]
    anyc = dev[s1] + dev[m1]
    rows.append({"reading": label, "any cluster % (25 +- 2.5)": anyc.mean(), "multiple % (1 +- 0.1)": dev[m1].mean(),
                 "any cluster at P11 % (30 +- 3)": anyc[dev.age == 11].mean(), "single only %": dev[s1].mean(),
                 "any cluster at P21 %": anyc[dev.age == 21].mean()})
tab = pd.DataFrame(rows)
print(tab.round(2).to_string(index=False))
t = 100 - dev.No_assemblies
print(f"The repository table, same datasets: any cluster {t.mean():.2f}%, multiple {dev.M_assemblies.mean():.2f}%, "
      f"at P11 {t[dev.age == 11].mean():.2f}%")

**Result: not reproduced.** Frames in any event cluster are 13.6% (10 runs, code test; 13.3 to
13.9% under the others), in multiple clusters 0.31%, and at P11 21.6%, against 25%, 1% and 30%. The
repository table gives 13.46%, 0.35% and 21.62% on the same datasets, so the measure agrees with the
paper's table and not with the numbers in its Discussion.

## Summary

| Claim | Outcome |
|---|---|
| quiescent-window-link | partially reproduced (87.7% and 58.3% of rows matched) |
| fig1d-event-frequency-decline | reproduced |
| fig1f-synchrony-decline | reproduced |
| figs2f-pearson-decline-females | reproduced |
| figs2f-pearson-l23-above-l4-p11 | reproduced |
| figs2g-mi-decline, figs2g-mi-male-above-female-p21, figs2g-mi-l4-below-l23, figs4b-mi-trajectory, figs4b-mi-sex-p21 | blocked: methods underspecified |
| fig2e-subnetworks-present-same-number | reproduced (assumed) |
| fig2d-silhouette-increase | reproduced |
| figs3c-tcc-inside-above-outside | reproduced |
| fig2f-tcc-increase-l23 | not reproduced (assumed) |
| figs3d-neuron-number-not-related | reproduced (assumed) |
| figs3e-time-interval-not-related | reproduced (assumed) |
| fig3a-single-cluster-decline | reproduced |
| fig3b-non-cluster-increase | reproduced (assumed) |
| fig3c-multiple-cluster-unchanged | reproduced (assumed) |
| fig3ef-subnetwork-size-unchanged | reproduced (assumed) |
| fig3g-centroid-distances-decrease | reproduced |
| fig4a-event-frequency-trajectory | not reproduced |
| fig4b-synchrony-gradual-decline | reproduced |
| figs4a-pearson-trajectory | reproduced |
| fig4c-subnetwork-number-transient-p15 | reproduced (assumed) |
| fig4d-silhouette-gradual-increase | partially reproduced |
| fig4e-tcc-maximum-earlier-l23 | not reproduced |
| fig4e-tcc-l4-below-l23-p15 | reproduced (assumed) |
| fig4fgh-configuration-trajectory | not reproduced (assumed) |
| figs4-distance-trajectory | not reproduced (assumed) |
| fig5cd-cno-reduces-events-synchrony | reproduced |
| figs6ab-cno-correlation-unchanged | not reproduced |
| fig5e-cno-silhouette-increase | reproduced |
| figs6c-cno-subnetwork-number-unchanged | not reproduced |
| figs6d-cno-tcc-females | reproduced |
| fig5fgh-cno-configuration | not reproduced (assumed) |
| figs6efg-cno-topography-unchanged | not reproduced (assumed) |
| figs7a-events-clustered | not reproduced |
| figs7-method-comparison, figs7-silhouette-increase-all-methods, figs7-subnetwork-number-stable, figs8-configuration-all-methods | blocked (not run in this run's time) |
| figs5e-mostly-glutamatergic, figs5f-hm3dq-mostly-gabaergic | blocked: data missing |
| figs9-no-sex-difference-gabaergic-input | reproduced (histology_figs9.ipynb) |
| figs9-layer-differences | partially reproduced (histology_figs9.ipynb) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**The assumption.** Where the readings of the open details O2 to O4 give different outcomes, the
outcome is taken under the code's readings (10 k-means runs, the 1 - 0.05/NCl participation test,
Step7's synchrony threshold, all ROIs as denominator), because the paper's own per-dataset tables
agree with these readings and not with the Methods' (data check: non-cluster ROIs 25.7% with the
code's test, 9.1% with the Methods', 25.0% in the table; synchrony 1.60 and 0.91 against 1.45).
The outcomes marked "assumed" rest on this; every reading is shown in its section.

**How the analysis differs from the paper's.** The quiescent windows come from the repository's
tables, not from behavioural video, which the dandiset does not hold. 387 developmental datasets of
30 subjects and 56 DREADD datasets of 7 subjects are used, against the tables' 453 and 96 (three
subjects and some fields are not in the dandiset; 12 files with sex U are left out). Events are the
dataset's `ca_events`. The k-means is scikit-learn's on an exact lower-dimensional form of the
covariance matrix, and shuffle thresholds of the participation tests are exact hypergeometric
quantiles. The cluster significance test of `Modol_script.m` (1,000 shuffles), whose result the
later steps do not use, is not run. In T1 and T2, likelihood-ratio tests replace Kenward-Roger tests,
and bootstrap replicates refit the fixed effects by generalized least squares with the variance
components held at the original fit.

**What was found about the dataset.** It holds 32 of the paper's subjects (three mice of litter
C57-C4-3 and several fields are missing); it has no behavioural video or quiescent windows, though
its keywords name behaviour and pose estimation; one file has age P190D (its session date is wrong);
two subjects have sex U; `ca_events` holds the events of the paper's Step2 at non-zero frames, with
values that are not described; the CNO condition is coded in the session description.

In [ ]:
print(f"notebook ran in {(time.time() - started)/60:.1f} min")